# Recomendação personalizada com fatoração de matriz

## Objetivo

Treinar o primeiro modelo personalizado do projeto e compará-lo ao baseline de popularidade. A fatoração de matriz aprende vetores latentes de usuários e filmes a partir dos ratings explícitos.

A seleção de hiperparâmetros usa somente a validação. O teste permanece intocado até a avaliação final.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data.loaders import load_movies, load_ratings
from data.splitting import temporal_leave_two_out
from evaluation.metrics import evaluate_top_k, rmse
from models.matrix_factorization import MatrixFactorizationRecommender
from models.popularity import PopularityRecommender

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

TOP_K = 10
RELEVANCE_THRESHOLD = 4.0

## Dados e split temporal

In [ ]:
raw_data_dir = PROJECT_ROOT / "data" / "raw"
ratings = load_ratings(raw_data_dir)
movies = load_movies(raw_data_dir)
train, validation, test = temporal_leave_two_out(ratings)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "users": [
            train["user_id"].nunique(),
            validation["user_id"].nunique(),
            test["user_id"].nunique(),
        ],
    }
)

## Referência: popularidade na validação

O mesmo baseline anterior é agora consumido a partir de `src/models`, sem duplicar sua implementação no notebook.

In [ ]:
popularity_model = PopularityRecommender().fit(train)
popularity_validation, _, _ = evaluate_top_k(
    validation,
    popularity_model.recommend,
    popularity_model.catalog_size,
    k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)
popularity_validation.to_frame()

## Experimentos de fatoração de matriz

Testamos configurações pequenas para entender o efeito da dimensão latente e da regularização. `random_state` fixa a inicialização e a ordem do SGD para tornar os resultados reproduzíveis.

In [ ]:
experiment_configs = [
    {
        "name": "mf_10",
        "n_factors": 10,
        "learning_rate": 0.01,
        "regularization": 0.05,
        "epochs": 6,
        "random_state": 42,
    },
    {
        "name": "mf_20",
        "n_factors": 20,
        "learning_rate": 0.01,
        "regularization": 0.05,
        "epochs": 6,
        "random_state": 42,
    },
    {
        "name": "mf_20_reg",
        "n_factors": 20,
        "learning_rate": 0.01,
        "regularization": 0.10,
        "epochs": 6,
        "random_state": 42,
    },
]
experiment_configs

In [ ]:
experiment_rows = []
fitted_models = {}

for config in experiment_configs:
    name = config["name"]
    model_params = {key: value for key, value in config.items() if key != "name"}
    model = MatrixFactorizationRecommender(**model_params).fit(train)
    validation_predictions = model.predict_pairs(validation)
    ranking_summary, _, _ = evaluate_top_k(
        validation,
        model.recommend,
        model.catalog_size,
        k=TOP_K,
        relevance_threshold=RELEVANCE_THRESHOLD,
    )

    experiment_rows.append(
        {
            **config,
            "validation_rmse": rmse(
                validation["rating"].to_numpy(float), validation_predictions
            ),
            "precision@10": ranking_summary["precision@10"],
            "recall@10": ranking_summary["recall@10"],
            "coverage@10": ranking_summary["catalog_coverage@10"],
            "final_train_rmse": model.history.train_rmse[-1],
        }
    )
    fitted_models[name] = model
    print(f"Concluído: {name}")

experiment_results = pd.DataFrame(experiment_rows).sort_values(
    ["recall@10", "validation_rmse"], ascending=[False, True]
)
experiment_results

## Comparação com o baseline

A fatoração é avaliada tanto como preditor de ratings (RMSE) quanto como rankeador (Recall/Precision/Coverage). Um bom RMSE não garante automaticamente um bom Top-10.

In [ ]:
comparison = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    "name": "popularity",
                    "validation_rmse": float("nan"),
                    "precision@10": popularity_validation["precision@10"],
                    "recall@10": popularity_validation["recall@10"],
                    "coverage@10": popularity_validation["catalog_coverage@10"],
                }
            ]
        ),
        experiment_results[
            ["name", "validation_rmse", "precision@10", "recall@10", "coverage@10"]
        ],
    ],
    ignore_index=True,
)
comparison

## Seleção e avaliação final

A configuração é escolhida por Recall@10 na validação, usando RMSE como desempate. Em seguida, ela é treinada novamente com treino + validação e avaliada uma única vez no teste.

In [ ]:
best_name = experiment_results.iloc[0]["name"]
best_config = next(config for config in experiment_configs if config["name"] == best_name)
best_params = {key: value for key, value in best_config.items() if key != "name"}

train_validation = pd.concat([train, validation], ignore_index=True)
final_model = MatrixFactorizationRecommender(**best_params).fit(train_validation)
test_predictions = final_model.predict_pairs(test)
test_ranking_summary, test_user_metrics, test_recommendations = evaluate_top_k(
    test,
    final_model.recommend,
    final_model.catalog_size,
    k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

final_metrics = pd.concat(
    [
        pd.Series({"model": best_name, "test_rmse": rmse(test["rating"].to_numpy(float), test_predictions)}),
        test_ranking_summary,
    ]
)
final_metrics.to_frame(name="value")

## Curva de treinamento

A queda do RMSE de treino confirma que o SGD está ajustando os fatores. Uma queda contínua no treino não prova generalização; por isso a escolha foi feita na validação.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    range(1, len(final_model.history.train_rmse) + 1),
    final_model.history.train_rmse,
    marker="o",
)
ax.set(
    title=f"Curva de treinamento — {best_name}",
    xlabel="Época",
    ylabel="RMSE de treino",
)
plt.show()

## Inspeção da personalização

Listas de usuários diferentes devem variar além da simples remoção de itens vistos, pois cada usuário possui seu próprio vetor latente.

In [ ]:
sample_users = test_user_metrics["user_id"].head(3).tolist()
for user_id in sample_users:
    movie_ids = test_recommendations[user_id]
    ranked = pd.DataFrame(
        {"rank": range(1, len(movie_ids) + 1), "movie_id": movie_ids}
    ).merge(movies, on="movie_id", how="left")
    print(f"\nUsuário {user_id}")
    display(ranked[["rank", "title", "genres"]])

## Conclusões

- A fatoração de matriz cria recomendações personalizadas por meio de embeddings de usuários e filmes.
- A seleção foi feita exclusivamente na validação; o teste foi usado apenas para a estimativa final.
- RMSE e Recall@10 medem comportamentos diferentes e devem ser analisados em conjunto.
- A próxima evolução é persistir o modelo e os mapas de IDs, automatizar o pipeline e comparar a fatoração com uma rede neural de embeddings.